In [ ]:
import cudf


In [2]:
INPUT_FILE = "/Users/eric/Desktop/data_cleaning/nypd.csv"

In [3]:
OUTPUT_FILE = "/Users/eric/Desktop/data_cleaning/nypd_clean.csv"

In [ ]:
# GPU-accelerated CSV read with cuDF
df = cudf.read_csv(INPUT_FILE)


In [5]:
df.columns

Index(['CMPLNT_NUM', 'ADDR_PCT_CD', 'BORO_NM', 'CMPLNT_FR_DT', 'CMPLNT_FR_TM',
       'CMPLNT_TO_DT', 'CMPLNT_TO_TM', 'CRM_ATPT_CPTD_CD', 'HADEVELOPT',
       'HOUSING_PSA', 'JURISDICTION_CODE', 'JURIS_DESC', 'KY_CD', 'LAW_CAT_CD',
       'LOC_OF_OCCUR_DESC', 'OFNS_DESC', 'PARKS_NM', 'PATROL_BORO', 'PD_CD',
       'PD_DESC', 'PREM_TYP_DESC', 'RPT_DT', 'STATION_NAME', 'SUSP_AGE_GROUP',
       'SUSP_RACE', 'SUSP_SEX', 'TRANSIT_DISTRICT', 'VIC_AGE_GROUP',
       'VIC_RACE', 'VIC_SEX', 'X_COORD_CD', 'Y_COORD_CD', 'Latitude',
       'Longitude', 'Lat_Lon', 'New Georeferenced Column'],
      dtype='object')

In [6]:
df.head(5)

,CMPLNT_NUM,ADDR_PCT_CD,BORO_NM,CMPLNT_FR_DT,CMPLNT_FR_TM,CMPLNT_TO_DT,CMPLNT_TO_TM,CRM_ATPT_CPTD_CD,HADEVELOPT,HOUSING_PSA,...,TRANSIT_DISTRICT,VIC_AGE_GROUP,VIC_RACE,VIC_SEX,X_COORD_CD,Y_COORD_CD,Latitude,Longitude,Lat_Lon,New Georeferenced Column
0,298784667,114,QUEENS,12/09/2024,04:37:00,NaN,(null),COMPLETED,(null),NaN,...,NaN,UNKNOWN,UNKNOWN,E,1016307,227998,40.792430,-73.884228,"(40.79243, -73.884228)",POINT (-73.884228 40.79243)
1,308328240,41,BRONX,06/18/2025,16:00:00,NaN,(null),COMPLETED,(null),NaN,...,NaN,UNKNOWN,UNKNOWN,E,1017940,232184,40.803914,-73.878308,"(40.803914, -73.878308)",POINT (-73.878308 40.803914)
2,314788366,40,BRONX,10/17/2025,22:00:00,NaN,(null),COMPLETED,(null),NaN,...,NaN,18-24,BLACK HISPANIC,F,1005028,234516,40.810352,-73.924942,"(40.8103518634571, -73.924942325642)",POINT (-73.924942325642 40.8103518634571)
3,307568863,41,BRONX,03/24/2025,01:00:00,NaN,(null),COMPLETED,(null),NaN,...,NaN,18-24,WHITE,F,1013037,236657,40.816206,-73.896001,"(40.8162058439227, -73.8960011932583)",POINT (-73.8960011932583 40.8162058439227)
4,311408135H1,42,BRONX,08/18/2025,22:34:00,NaN,(null),COMPLETED,(null),NaN,...,NaN,25-44,BLACK,M,1009748,240283,40.826169,-73.907868,"(40.826169, -73.907868)",POINT (-73.907868 40.826169)


In [7]:
# Normalize column names on GPU
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace("%", "percent", regex=False)
    .str.replace(r"[()]", "_", regex=True)
    .str.replace(r"[^\w]", "_", regex=True)
    .str.replace(r"_+", "_", regex=True)
    .str.strip("_")
)


In [8]:
# Remove empty / duplicate rows
df = df.dropna(how="all")
df = df.drop_duplicates()

# Clean string columns on GPU
# NOTE: cuDF "string" columns may not be reported as dtype "object".
str_cols = df.select_dtypes(include=["object", "str"]).columns

# Normalize common "missing" sentinels to real nulls
missing_tokens = {
    "": None,
    "na": None,
    "n/a": None,
    "null": None,
    "(null)": None,
    "none": None,
}

for col in str_cols:
    s = df[col].str.strip().str.lower()
    df[col] = s.replace(missing_tokens)



In [9]:
null_counts = df.isnull().sum()
accidents_nulls = cudf.DataFrame({
    "Column": null_counts.index,
    "Number of Null Values": null_counts.values,
})
accidents_nulls


,Column,Number of Null Values
0,cmplnt_num,0
1,addr_pct_cd,0
2,boro_nm,0
3,cmplnt_fr_dt,0
4,cmplnt_fr_tm,0
5,cmplnt_to_dt,26380
6,cmplnt_to_tm,0
7,crm_atpt_cptd_cd,0
8,hadevelopt,0
9,housing_psa,543632


In [10]:
# Drop columns that are not meaningful for analysis or have too many missing values
columns_to_drop = [
    "cmplnt_num", "addr_pct_cd", "prem_typ_desc", "cmplnt_to_dt", "housing_psa",
    "pd_desc", "transit_district", "pd_cd", "cmplnt_to_tm", "hadevelopt",
    "jurisdiction_code", "juris_desc", "ky_cd", "loc_of_occur_desc", "rpt_dt",
    "station_name", "x_coord_cd", "y_coord_cd", "lat_lon", "new_georeferenced_column"
]

existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns=existing_columns_to_drop)


In [11]:
df.head(5)

,boro_nm,cmplnt_fr_dt,cmplnt_fr_tm,crm_atpt_cptd_cd,law_cat_cd,ofns_desc,parks_nm,patrol_boro,susp_age_group,susp_race,susp_sex,vic_age_group,vic_race,vic_sex,latitude,longitude
0,QUEENS,12/09/2024,04:37:00,COMPLETED,FELONY,ARSON,(null),PATROL BORO QUEENS NORTH,18-24,WHITE HISPANIC,M,UNKNOWN,UNKNOWN,E,40.792430,-73.884228
1,BRONX,06/18/2025,16:00:00,COMPLETED,FELONY,ARSON,(null),PATROL BORO BRONX,18-24,WHITE HISPANIC,M,UNKNOWN,UNKNOWN,E,40.803914,-73.878308
2,BRONX,10/17/2025,22:00:00,COMPLETED,FELONY,RAPE,(null),PATROL BORO BRONX,25-44,BLACK,M,18-24,BLACK HISPANIC,F,40.810352,-73.924942
3,BRONX,03/24/2025,01:00:00,COMPLETED,FELONY,RAPE,(null),PATROL BORO BRONX,UNKNOWN,UNKNOWN,M,18-24,WHITE,F,40.816206,-73.896001
4,BRONX,08/18/2025,22:34:00,COMPLETED,FELONY,MURDER & NON-NEGL. MANSLAUGHTER,(null),PATROL BORO BRONX,45-64,BLACK,M,25-44,BLACK,M,40.826169,-73.907868


In [ ]:
df.to_csv(OUTPUT_FILE, index=False)


### cuDF notes
- This notebook now uses **NVIDIA cuDF** for GPU-accelerated CSV loading, string cleaning, null analysis, column dropping, and CSV export.
- It requires a CUDA-capable NVIDIA GPU with RAPIDS/cuDF installed.

